# RAGF Ontology Auditor · Quickstart

End-to-end tour of the auditor against a live Neo4j sandbox:

1. Connect to the sandbox.
2. Load `examples/fintech_minimal.yaml` (the clean reference).
3. Inspect the projected graph.
4. Run the auditor and view the report as a pandas DataFrame.
5. Replay with `examples/fintech_seeded_faults.yaml` and see each CC fire.
6. For every failing CC, re-execute its `.cypher` inline so the evidence is reproducible without leaving the notebook.

## Prereqs

```bash
make up              # start the Neo4j 5.26 + GDS sandbox
pip install -e ".[dev]" pandas    # pandas is needed by this notebook
export AUDITOR_HMAC_KEY="dev-only-not-for-production"
```

Run the cells in order. The whole notebook completes in well under a minute.

## 1 · Connect

In [ ]:
import os
from pathlib import Path

import pandas as pd
import yaml
from neo4j import GraphDatabase

from harness_auditor import (
    Ontology,
    build_report,
    load,
    packaged_queries_dir,
    run_all,
)
from harness_auditor._environment import probe_gds_version, probe_neo4j_version

URI = os.environ.get("NEO4J_URI", "bolt://127.0.0.1:7687")
USER = os.environ.get("NEO4J_USER", "neo4j")
PWD = os.environ.get("NEO4J_PASSWORD", "auditor_local_only")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLES = REPO_ROOT / "examples"
QUERIES = packaged_queries_dir()

# Single source of truth for the query parameters the bundled CCs read.
# Mirrored from cli.py defaults; if the CLI defaults change, update this
# dict in the same commit so the notebook does not silently drift.
#
# Neo4j 5.x requires every parameter referenced by a query to be present
# in the parameters dict, even if the query envelopes it in coalesce(...);
# omitting one raises Neo.ClientError.Statement.ParameterMissing. CC-06
# reads $coverage_threshold; CC-11 reads $threshold_ratio and
# $cc11_hysteresis.
QUERY_PARAMS: dict[str, float] = {
    "threshold_ratio": 1.3,       # CC-11 base ratio
    "cc11_hysteresis": 0.05,      # CC-11 stability margin (effective threshold = ratio + hysteresis)
    "coverage_threshold": 0.85,   # CC-06 per-verb regulatory coverage floor
}

driver = GraphDatabase.driver(URI, auth=(USER, PWD))
driver.verify_connectivity()
print(f"Connected to {URI}")

## 2 · Load the clean ontology

In [ ]:
def load_yaml(path: Path) -> Ontology:
    return Ontology.model_validate(yaml.safe_load(path.read_text(encoding="utf-8")))


clean = load_yaml(EXAMPLES / "fintech_minimal.yaml")
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    counts = load(session, clean)
    neo4j_v = probe_neo4j_version(session)
    gds_v = probe_gds_version(session)

print(f"Neo4j: {neo4j_v} · GDS: {gds_v}")
pd.Series(counts, name="count").to_frame()

## 3 · Inspect the graph

A quick table view of the verb→regulation projection — the substrate every CC reads from.

In [ ]:
def runq(cypher: str, **params) -> pd.DataFrame:
    with driver.session() as session:
        rows = [r.data() for r in session.run(cypher, **params)]
    return pd.DataFrame(rows)


runq(
    """
    MATCH (v:Verb)-[:MUST_SATISFY]->(r:Regulation)
    OPTIONAL MATCH (c:Constraint)-[:HAS_CONSTRAINT_OF]->(v)
    OPTIONAL MATCH (c)-[:REFERENCES]->(rc:Regulation {code: r.code})
    RETURN v.name AS verb,
           v.risk_level AS risk,
           v.min_amm_level AS amm,
           r.code AS must_satisfy,
           collect(DISTINCT c.name) AS enforcing_constraints
    ORDER BY verb, must_satisfy
    """
)

## 4 · Run the auditor

In [ ]:
def run_audit(domain_name: str, domain_version: str) -> pd.DataFrame:
    with driver.session() as session:
        results = run_all(session, QUERIES, query_params=QUERY_PARAMS)
    report = build_report(
        ontology_sha256="0" * 64,
        domain=domain_name,
        domain_version=domain_version,
        criteria=results,
        hmac_key_present=True,
        neo4j_version=neo4j_v,
        gds_version=gds_v,
    )
    df = pd.DataFrame(
        [
            {
                "id": c.criterion_id,
                "name": c.name,
                "status": c.status.value,
                "severity": c.severity.value,
                "latency_ms": round(c.latency_ms, 1),
                "message": c.message,
            }
            for c in report.criteria
        ]
    )
    print(f"verdict: {report.certification_status.value}")
    return df


run_audit(clean.domain.name, clean.domain.version)

## 5 · Replay against the seeded-faults ontology

Same pipeline, dirtier input. Every defect in `fintech_seeded_faults.yaml` is intentional — see the file's header comment for the catalogue.

In [ ]:
broken = load_yaml(EXAMPLES / "fintech_seeded_faults.yaml")
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    load(session, broken)

broken_df = run_audit(broken.domain.name, broken.domain.version)
broken_df

## 6 · Inspect evidence for every failing CC

For each CC in `FAIL`, re-run its bundled `.cypher` inline so the evidence is reproducible without leaving the notebook.

In [ ]:
from harness_auditor.runner import REGISTRY, _split_cypher_statements

fail_ids = list(broken_df[broken_df["status"].isin(["FAIL", "ERROR"])]["id"])
registry_by_id = {d.criterion_id: d for d in REGISTRY}

for cid in fail_ids:
    definition = registry_by_id.get(cid)
    if definition is None:
        continue
    query_path = (definition.query_dir or QUERIES) / definition.query_file
    cypher = query_path.read_text(encoding="utf-8")
    print(f"\n=== {cid} · {definition.name} ===")
    print(f"(query: {query_path.relative_to(REPO_ROOT)})")
    # Re-use the runner's splitter so comments-with-semicolons inside the
    # .cypher header are handled correctly; evidence comes from the last
    # statement, exactly as run_all() does it. Pass the full QUERY_PARAMS
    # dict so every parameter referenced by any CC (coverage_threshold,
    # cc11_hysteresis, threshold_ratio) is present — Neo4j 5.x raises
    # ParameterMissing on referenced-but-absent parameters even when the
    # query wraps them in coalesce(...).
    statements = _split_cypher_statements(cypher)
    with driver.session() as session:
        for setup in statements[:-1]:
            session.run(setup, **QUERY_PARAMS)
        df = pd.DataFrame(
            [r.data() for r in session.run(statements[-1], **QUERY_PARAMS)]
        )
    if df.empty:
        print("  (no rows — query passed)")
    else:
        display(df.head(10))


## 7 · Cleanup

In [ ]:
driver.close()
print("driver closed")

When you are done with the notebook, tear down the sandbox from the shell:

```bash
make down
```